# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`
This notebook provides a step-by-step guide for exploring this FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described by a Croissant schema at:

https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Instantiate the Dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print Dataset Title and Description
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review the available record sets, fields, and their `@id`s.

The dataset comprises three main tables (record sets):
  * Table 1: Clinical features of each patient
  * Table 2: Laboratory results and imaging
  * Table 3: CYP21A2 genotype and variant annotations

Let's enumerate them based on their `@id` and examine their structure.

In [ ]:
# List available record sets in the metadata
record_set_objs = list(dataset.metadata.record_sets)
record_set_ids = [rs.id for rs in record_set_objs]

print("Record set `@id`s:")
for rs in record_set_objs:
    print(f"- {rs.id}: {rs.name}")

# Display the fields (columns) of each record set by `@id`
for rs in record_set_objs:
    print(f"\nRecordSet {rs.id} ({rs.name}):")
    for field in rs.fields:
        print(f"  Field @id: {field.id}, name: {field.name}, dtype: {field.data_type}")

### Show example records from each record set
We'll use the `@id`s for referencing record sets as required.

In [ ]:
# Preview a sample record from each record set by `@id`
for rs_id in record_set_ids:
    print(f"\nSample record from RecordSet @id: {rs_id}")
    try:
        for record in dataset.records(record_set=rs_id):
            print(record)
            break  # Only show first record
    except Exception as e:
        print(f"Could not load records for {rs_id}: {str(e)}")

## 3. Data Extraction

Load all records from each record set (by `@id`) into DataFrames for structured analysis.

In [ ]:
# Prepare to extract all data for each record set
dataframes = {}

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"\nColumns in DataFrame for @id={record_set_id}:")
        print(df.columns.tolist())
        print("First 3 rows:")
        print(df.head(3))
    except Exception as e:
        print(f"Failed to load {record_set_id}: {str(e)}")

## 4. Exploratory Data Analysis (EDA)

Let's process and normalize a numeric field, filter records, and optionally group by additional key fields. All fields are referenced using their `@id`.

In [ ]:
# Choose a record set with numeric data (eg. hormone levels from 'Table 2')
clinical_rs_name = "Table 2: Hormone levels and imaging"

# Use its `@id` from previously printed record_set_ids
clinical_rs_id = record_set_ids[1]  # Assuming second RecordSet is Table 2
df = dataframes[clinical_rs_id]

# List candidate numeric fields
numeric_fields = [col for col in df.columns if df[col].dtype in ['float64', 'int64'] or pd.api.types.is_numeric_dtype(df[col])]
print(f"Numeric fields in {clinical_rs_id}: {numeric_fields}")

# Pick hormone level for filtering (e.g., 'cr:field-serum17OHP')
numeric_field_id = None
for col in df.columns:
    if '17OHP' in col or 'cr:field-serum17OHP' in col:
        numeric_field_id = col
        break
if numeric_field_id is None and numeric_fields:
    numeric_field_id = numeric_fields[0]  # fallback to first numeric field

print(f"Using numeric field: {numeric_field_id}")

# Set a filter threshold
threshold = 10
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records where {numeric_field_id} > {threshold}:")
print(filtered_df.head())

# Normalize the numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized values for {numeric_field_id}:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try grouping by a categorical field (e.g., imaging result @id)
group_field = None
for col in df.columns:
    if 'ctResult' in col or 'ultrasound' in col or 'cr:field-adrenalCTResult' in col:
        group_field = col
        break

if group_field and group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field).mean()
    print(f"Grouped mean results by {group_field}:")
    print(grouped_df.head())

## 5. Visualization

Visualize hormone level distribution and relationships. All axes and legends reference columns via their `@id`s.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram for the numeric hormone field in clinical_rs_id DataFrame
plt.figure(figsize=(7, 5))
sns.histplot(df[numeric_field_id], bins=5, kde=True)
plt.title(f"Distribution of {numeric_field_id} in RecordSet @id={clinical_rs_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Patient count")
plt.show()

# If available, scatter plot of hormone vs. age in another record set (Table 1)
demography_rs_id = record_set_ids[0]  # Table 1
demography_df = dataframes[demography_rs_id]

age_field_id = None
for col in demography_df.columns:
    if 'age' in col or 'cr:field-ageAtDiagnosis' in col:
        age_field_id = col
        break
if age_field_id:
    combined = df.copy()
    combined[age_field_id] = demography_df[age_field_id] if age_field_id in demography_df.columns else None
    plt.figure(figsize=(7,5))
    sns.scatterplot(x=age_field_id, y=numeric_field_id, data=combined)
    plt.title(f"{numeric_field_id} vs. {age_field_id}")
    plt.xlabel(age_field_id)
    plt.ylabel(numeric_field_id)
    plt.show()

## 6. Conclusion

Through this notebook, we explored the structure and contents of the FAIR^2 clinical dataset using the `mlcroissant` library:

- Record sets and fields were referenced by their `@id`.
- Data extraction and overview identified key patient features, hormone levels, imaging, and genetic variant records.
- Numeric hormone fields were filtered and normalized, with groupings by imaging results.
- Visualizations illustrated data distributions and clinical correlations.

This approach demonstrates robust, reproducible FAIR data exploration conforming to Croissant schema standards, supporting further clinical or scientific analyses.